# Lumen on Kaggle

The code lives in the private GitHub repo; this notebook only clones it and runs `kaggle/run_stages.py`. Details: `kaggle/README.md`.

**Settings (right sidebar):**
- **Accelerator:** *None* for Stage 1 cells, *GPU T4* for Stage 2.
- **Internet:** on. **Persistence:** *Files only* or *Variables and Files* (keeps the 15 GB of data and partial runs between sessions).
- **Secrets (Add-ons → Secrets):** `GITHUB_TOKEN` (read access to the repo) and `HF_TOKEN` (write access to `cjitendr/lumen-predictions` and `cjitendr/lumen-adapter` only).

**Every session:** run **1 → 2 → 3** first (clone/update, setup, environment). Then jump to the stage you need.

In [ ]:
# Clone or update the private repo. The GitHub token never appears in a command line, a URL,
# notebook output, an exception, or on disk: git receives it only as an HTTP auth header through
# environment variables set for the git process alone (tests/test_notebook_clone.py enforces this).
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/JCHETAN26/lumen.git"
WORKDIR = os.environ.get("LUMEN_KAGGLE_WORKDIR", "/kaggle/working")


def _git(*args: str) -> None:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if not k.startswith("GIT_TRACE")}
    env.update(
        GIT_TERMINAL_PROMPT="0",  # fail instead of prompting for credentials
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}",
    )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True, cwd=WORKDIR)
    message = (result.stderr or result.stdout or "").strip()
    for secret in (token, basic):  # redact, in case git ever echoes the header
        message = message.replace(secret, "***")
    del token, basic, env
    if result.returncode != 0:
        raise RuntimeError(
            f"git {args[0]} failed (exit {result.returncode}): {message[-800:]}"
        ) from None


os.makedirs(WORKDIR, exist_ok=True)
if not os.path.isdir(os.path.join(WORKDIR, "lumen")):
    _git("clone", "--depth", "1", REPO_URL, "lumen")
else:
    _git("-C", "lumen", "pull", "--ff-only", REPO_URL, "main")
head = subprocess.run(
    ["git", "-C", os.path.join(WORKDIR, "lumen"), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
print("repo at", head)

In [ ]:
# 2. Setup: installs the pinned environment (CUDA PyTorch only when a GPU is attached). Idempotent.
%cd /kaggle/working/lumen
!bash kaggle/setup.sh

In [ ]:
# 3. Environment: records the machine, GPU, and library versions.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py env

## Stage 1 (done 2026-09-28): data and Mac sample. CPU only.

Skip these unless the data check in S2.0 says the zips are missing. Run them with **Accelerator: None**.

In [ ]:
# S1.1 Download + verify the pinned VizWiz files (~15 GB, resumable), build the splits.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py data

In [ ]:
# S1.2 Package the 300-image Mac sample (already built; prints 'exists' if so).
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py mac_sample

## Stage 2: zero-shot baselines (Accelerator: GPU T4)

Every session: cells **1 → 2 → 3**, then **S2.0 → S2.1 → S2.2**. Then run the next block you haven't finished; each run resumes where it stopped if the session ends. Prompts are chosen on a fixed 512-question dev subset (`dev_select`), never on val. Prefetching is on (2 workers; measured faster). Prediction files go only to the private Hugging Face dataset.

In [ ]:
# S2.0 Data present? Expect Annotations.zip, train.zip (~11.3 GB), val.zip (~3.5 GB).
# If missing: switch to Accelerator None and run S1.1 first (downloading on GPU wastes quota).
!ls -la /kaggle/working/lumen-data/

In [ ]:
# S2.1 Quota + Hugging Face token. The token is exported for the `!` commands below and
# never printed.
import os

import kaggle_secrets

GPU_HOURS_LEFT = 30.0  # <- set to your real remaining weekly quota (kaggle.com -> Settings)
os.environ["HF_TOKEN"] = kaggle_secrets.UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# S2.2 Storage check (seconds): both private HF repos exist, are private, and HF_TOKEN can write.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py storage_check

In [ ]:
# S2.3 Plan (free): prior GPU hours for SmolVLM2-500M's two runs. Loads nothing.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev_select --prompt both --dry-run --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split val --prompt dev-selected --dry-run --gpu-hours-left $GPU_HOURS_LEFT

### Quick checks (both passed on 2026-09-29/30; optional)

In [ ]:
# S2.4 Quick check 1 (~2-5 min): tiny random-weight stand-ins with the real processors on the
# GPU. Proves CUDA + data + pipeline. Written under baseline/stand-in/, never uploaded.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m lfm2.5-vl-450m --split dev --prompt abstain --stand-in --limit 16 --device cuda

In [ ]:
# S2.5 Quick check 2 (passed): 32 real dev questions, fp16, SmolVLM2-500M. Smoke only.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --limit 32 --gpu-hours-left $GPU_HOURS_LEFT

### SmolVLM2-500M (this week, first): ~0.36 h + ~1.15 h

In [ ]:
# S2.6 SmolVLM2-500M prompt choice: both prompts on the fixed 512-question dev subset.
# The first run also builds dev_select.txt (in the download folder's splits/) to be committed.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev_select --prompt both --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.7 SmolVLM2-500M on val with the prompt it scored best with on dev_select.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split val --prompt dev-selected --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.8 Summary. Send this output for review.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

In [ ]:
# S2.R Recovery (CPU, Accelerator: None; only if result files were lost with a session's disk).
# Rebuilds them from the predictions in the private HF dataset: re-scores, never re-answers.
# Needs cells 1 -> 2 and S2.1 (HF_TOKEN). Then download lumen-results and commit as usual.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py rebuild_baseline --model smolvlm2-500m
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

### LFM2.5-VL-450M and SmolVLM2-256M (this week, after S2.8 is reviewed): ~1.5 h + ~1.25 h

In [ ]:
# S2.9 Prompt choice for both on dev_select.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model lfm2.5-vl-450m smolvlm2-256m --split dev_select --prompt both --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.10 Val for both, each with its dev_select-chosen prompt.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model lfm2.5-vl-450m smolvlm2-256m --split val --prompt dev-selected --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.11 Summary.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

### Stage 3 prep: image resolution probe (this week, after S2.6 results are committed): ~1.7 h (prior) + ~0.5-0.8 h

SmolVLM2-500M zero-shot on **all 2,052 dev questions** at a long-edge cap of 2,048 (default), 1,536, 1,024, and 512 px (13 / 10 / 5 / 1 tiles of 64 tokens for a portrait photo). The 2,048 px arm is a full-dev run at the default resolution; if a partial `dev-abstain` predictions file is on disk from earlier checks, it resumes from it. The rule, fixed 2026-10-03 before any result: the smallest long edge whose paired 95% CI on its accuracy difference from the best arm stays within -2 points (AP reported, not decisive). Uses the prompt S2.6 chose (abstain); `resolution_choice` refuses if the runs used a different one.

In [ ]:
# S3.1 Plan (free). Prior: 0.60 + 0.49 + 0.30 + 0.28 = 1.68 GPU h; calibration re-checks each run.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --dry-run --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --dry-run --long-edge 1536 --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --dry-run --long-edge 1024 --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --dry-run --long-edge 512 --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S3.2 The probe: four full-dev runs (resumable). Run them in order; stop between runs if the
# quota gets tight, and the next session resumes.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --long-edge 1536 --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --long-edge 1024 --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --long-edge 512 --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S3.3 The choice (CPU, seconds; fetches predictions from the private HF dataset if they
# aren't on disk). Send this output for review, and commit
# results/stage3_prep/resolution_choice.json before S3.4.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py resolution_choice

In [ ]:
# S3.4 SmolVLM2-500M on val at the chosen resolution: the zero-shot baseline the fine-tune is
# compared against. Does nothing if the default 2,048 px was chosen (S2.7 already is that run).
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split val --prompt dev-selected --long-edge chosen --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

### Stage 3 prep: training speed test (after S3.3 is committed): ~0.4-0.6 h (prior)

200 LoRA steps (batch 4 x 2 accumulation = 8) on `train_fit` at the chosen resolution, fp16 with loss scaling: the first real run of the fp16 path. Records s/step (median, p90), peak GPU memory, tokens/s, the NaN/overflow counts, and the projected hours per epoch. Checkpoints stay on Kaggle's disk; nothing is published.

In [ ]:
# S3.5 Training speed test: plan (free), then the run. Send the output for review.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py train_speed --long-edge chosen --dry-run
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py train_speed --long-edge chosen --gpu-hours-left $GPU_HOURS_LEFT

### SmolVLM2-2.2B (next week, fresh quota): ~1.1 h + ~3.8 h

In [ ]:
# S2.12 SmolVLM2-2.2B: prompt choice on dev_select, then val. Run next week.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-2.2b --split dev_select --prompt both --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-2.2b --split val --prompt dev-selected --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

## Download

Download `/kaggle/working/lumen-results/` from the **Output** tab: result JSONs (aggregate scores + metadata), split ID lists, and the Mac sample archive. Prediction files are **not** in it: completed runs upload them to the private Hugging Face dataset, and each result records their Hub revision and sha256. Image zips and local prediction copies stay on Kaggle.